# Day 2 — MLOps in Production (Practice)

Companion notebook for [day2_materials.md](day2_materials.md).

## Contents
- **Lab 4** — Package and serve a model with FastAPI ([theory](day2_materials.md#module-5--packaging--serving-models-0900--1030))
- **Lab 5** — Tests for ML: data, model, behaviour ([theory](day2_materials.md#module-6--cicd-for-machine-learning-1045--1215))
- **Lab 6** — Drift detection (PSI + Evidently) ([theory](day2_materials.md#module-7--monitoring--drift-1315--1445))
- **Lab 7 — Capstone** — connect train → register → serve → monitor ([theory](day2_materials.md#module-8--industry-best-practices-governance-recap-1500--1630))

## Lab 4 — Package & serve a model

We train a model, persist it as an artifact, write a FastAPI service, and call it.

Reference: [Day 2, Module 5](day2_materials.md#module-5--packaging--serving-models-0900--1030)

In [ ]:
from pathlib import Path
import joblib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
feature_names = list(X.columns)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X_tr, y_tr)

joblib.dump({'model': model, 'features': feature_names}, ARTIFACT_DIR / 'model.joblib')
print('Saved', ARTIFACT_DIR / 'model.joblib')

Now write a tiny FastAPI service to disk.

In [ ]:
service_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Dict
import joblib, os

ARTIFACT = os.environ.get("MODEL_PATH", "artifacts/model.joblib")
bundle = joblib.load(ARTIFACT)
model, FEATURES = bundle["model"], bundle["features"]

app = FastAPI(title="breast-cancer-classifier", version="1.0")

class Payload(BaseModel):
    features: Dict[str, float]

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/ready")
def ready():
    return {"status": "ready", "n_features": len(FEATURES)}

@app.post("/predict")
def predict(p: Payload):
    missing = [f for f in FEATURES if f not in p.features]
    if missing:
        raise HTTPException(status_code=422, detail={"missing_features": missing})
    row = [[p.features[f] for f in FEATURES]]
    proba = float(model.predict_proba(row)[0, 1])
    return {"probability": proba, "label": int(proba >= 0.5)}
'''
Path('service.py').write_text(service_code)
print('Wrote service.py')

Start the service in a terminal **from the `day2/` folder**:

```bash
uvicorn service:app --host 127.0.0.1 --port 8000
```

Then run the cell below to call it.

In [ ]:
import requests

sample = X_te.iloc[0].to_dict()
try:
    r = requests.post('http://127.0.0.1:8000/predict', json={'features': sample}, timeout=2)
    print(r.status_code, r.json())
except Exception as e:
    print('Service not running yet?', e)

**Exercise 4.1** — Send a payload with a missing feature. Confirm a 422 error.

**Exercise 4.2** — Add an `X-Model-Version` response header to the service. Why is this useful for monitoring?

**Exercise 4.3 (optional)** — Write a `Dockerfile` (see [Day 2, §5.5](day2_materials.md#55-containers)) and build the image.

## Lab 5 — Tests for ML

Three tiers: **data tests**, **model quality tests**, **behavioural tests**.

Reference: [Day 2, Module 6](day2_materials.md#module-6--cicd-for-machine-learning-1045--1215)

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

# --- Data test --------------------------------------------------
def test_no_nulls():
    assert not X_tr.isna().any().any()

def test_label_balance():
    pos_rate = y_tr.mean()
    assert 0.2 < pos_rate < 0.8, f'label imbalance: {pos_rate:.2f}'

# --- Model quality test ----------------------------------------
def test_holdout_auc():
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    assert auc >= 0.95, f'AUC regressed: {auc:.3f}'

# --- Behavioural / invariance tests ----------------------------
def test_invariance_to_irrelevant_jitter():
    """Tiny noise on inputs must not flip predictions wildly."""
    base = X_te.iloc[:50].to_numpy()
    noisy = base + np.random.default_rng(0).normal(0, 1e-6, base.shape)
    p1 = model.predict_proba(base)[:, 1]
    p2 = model.predict_proba(noisy)[:, 1]
    assert np.max(np.abs(p1 - p2)) < 1e-3

for fn in [test_no_nulls, test_label_balance, test_holdout_auc, test_invariance_to_irrelevant_jitter]:
    fn(); print('PASS', fn.__name__)

**Exercise 5.1** — Convert the four tests into a `tests/test_model.py` file and run them with `pytest`.

**Exercise 5.2** — Add a **directional** test: increasing `mean radius` (a known risk factor) should not *decrease* the predicted probability.

**Exercise 5.3** — Add a **fairness-style** test: AUC on the *bottom-quartile-`mean area`* slice is no more than 5 points below the overall AUC.

## Lab 6 — Drift detection

We synthesise a drifted production dataset, then detect the drift two ways: a hand-rolled **Population Stability Index (PSI)** and an **Evidently** report.

Reference: [Day 2, Module 7](day2_materials.md#module-7--monitoring--drift-1315--1445)

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
reference = X_tr.copy()

# Simulate a 'production' batch where 'mean radius' shifts up by 30%
production = X_te.copy()
production['mean radius'] = production['mean radius'] * 1.3 + rng.normal(0, 0.5, len(production))

def psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e_counts, _ = np.histogram(expected, bins=edges)
    a_counts, _ = np.histogram(actual,   bins=edges)
    e_pct = np.clip(e_counts / e_counts.sum(), 1e-6, None)
    a_pct = np.clip(a_counts / a_counts.sum(), 1e-6, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

report = pd.DataFrame({
    'feature': feature_names,
    'psi': [psi(reference[c].to_numpy(), production[c].to_numpy()) for c in feature_names],
}).sort_values('psi', ascending=False)

def severity(p):
    if p < 0.1: return 'ok'
    if p < 0.25: return 'warn'
    return 'alert'
report['status'] = report['psi'].map(severity)
report.head(10)

In [ ]:
# Optional: rich HTML report with Evidently
try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    rep = Report(metrics=[DataDriftPreset()])
    rep.run(reference_data=reference, current_data=production)
    rep.save_html('drift_report.html')
    print('Wrote drift_report.html — open it in a browser.')
except Exception as e:
    print('Evidently optional step skipped:', e)

**Exercise 6.1** — How does the drifted feature affect the model's prediction distribution? Plot histograms of `predict_proba` on `reference` vs `production`.

**Exercise 6.2** — Define an alert rule: e.g. *"if any feature has PSI > 0.25, page the on-call."* Discuss false positives.

**Exercise 6.3 (concept drift)** — Flip 20% of `y_te` labels and recompute AUC. Which kind of drift is this and why is it harder to detect in real time?

## Lab 7 (Capstone) — Connect the pieces

Goal: end-to-end mini system tying together the techniques from both days.

Reference: [Day 2, Module 8](day2_materials.md#module-8--industry-best-practices-governance-recap-1500--1630)

**Specification**

1. **Train** the breast-cancer classifier with MLflow logging (params, metrics, model, git SHA tag).
2. **Validate** the dataset (use Day 1 Lab 3 functions).
3. **Gate**: only register the model in MLflow if `auc >= 0.97` AND data validation passed.
4. **Save** the latest registered model artifact to `artifacts/model.joblib` so the FastAPI service from Lab 4 picks it up.
5. **Smoke-test** the service endpoint after "deploying".
6. **Monitor**: score the production-like batch from Lab 6 and emit a PSI + prediction-distribution report.

Use the cell below as a scaffold.

In [ ]:
import subprocess, json, mlflow, mlflow.sklearn

AUC_GATE = 0.97
EXPERIMENT = 'capstone-breast-cancer'

def git_sha() -> str:
    try:
        return subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
    except Exception:
        return 'unknown'

def capstone_pipeline():
    # 1. train + log
    mlflow.set_tracking_uri('file:./mlruns')
    mlflow.set_experiment(EXPERIMENT)
    with mlflow.start_run() as run:
        mlflow.set_tag('git_sha', git_sha())
        params = {'n_estimators': 100, 'max_depth': 5}
        mlflow.log_params(params)
        m = RandomForestClassifier(random_state=42, **params).fit(X_tr, y_tr)
        auc = float(roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]))
        mlflow.log_metric('auc', auc)
        mlflow.sklearn.log_model(m, name='model')
        run_id = run.info.run_id
    print(f'run_id={run_id} auc={auc:.4f}')

    # 2/3. quality gate
    if auc < AUC_GATE:
        print(f'GATE FAILED: AUC {auc:.4f} < {AUC_GATE}')
        return

    # 4. publish artifact for the service
    joblib.dump({'model': m, 'features': feature_names}, ARTIFACT_DIR / 'model.joblib')
    print('Published artifact to artifacts/model.joblib')

    # 5. smoke-test (only if service is running)
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=1)
        print('Service health:', r.status_code, r.json())
    except Exception as e:
        print('Service not running — restart uvicorn to load the new model. Skipped smoke test.')

    # 6. monitoring snapshot
    psi_report = pd.DataFrame({
        'feature': feature_names,
        'psi': [psi(reference[c].to_numpy(), production[c].to_numpy()) for c in feature_names],
    }).sort_values('psi', ascending=False).head(5)
    print('\nTop drifted features:')
    print(psi_report.to_string(index=False))

capstone_pipeline()

### Stretch goals

- Replace the `joblib.dump` step with `mlflow.register_model(...)` and load the model in the service via the MLflow URI `models:/breast-cancer-classifier/Production`.
- Add a GitHub Actions workflow (`.github/workflows/ml-ci.yml`) that runs `pytest` on every push (see [Day 2, §6.4](day2_materials.md#64-github-actions-sketch)).
- Schedule the capstone pipeline to run nightly with Prefect or cron.

## Course wrap-up

You now have running code for every stage of an MLOps lifecycle:

| Stage | Where |
|-------|-------|
| Reproducibility | Day 1 Lab 1 |
| Tracking | Day 1 Lab 2 |
| Pipelines + validation | Day 1 Lab 3 |
| Packaging + serving | Day 2 Lab 4 |
| Tests | Day 2 Lab 5 |
| Monitoring | Day 2 Lab 6 |
| End-to-end | Day 2 Lab 7 |

Take the [non-negotiables list](day2_materials.md#85-the-non-negotiables-one-slide) back to your team — they are the cheapest 80% of the value of MLOps.